In [23]:
# root output dir

import os

import shutil

import torch

import numpy as np

import json


xai_output = "/mnt/abka03/Projects/xl-vlms/outputs/1000_imagenet_cdgl"
front_end_output = f"{xai_output}/frontend"
prototypes_path = f"{front_end_output}/prototypes"
image_crop_path = f"{front_end_output}/crops"
concept_pth = f"{xai_output}/concept/snmf/combined_concept_snmf_raw.pth"

def tensor_to_list(obj):
    """Recursively convert torch.Tensor → list (via numpy), and handle nested structures."""
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy().tolist()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: tensor_to_list(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [tensor_to_list(v) for v in obj]
    else:
        return obj

def get_relative_crop_path(img_path, crop_root):
    """Truncate the path at 'crops', so the returned path is always after crops/."""
    # Find 'crops' in the path and return everything after it
    parts = os.path.normpath(img_path).split(os.sep)
    if 'crops' in parts:
        crops_idx = parts.index('crops')
        return os.path.join(*parts[crops_idx+1:])
    else:
        return os.path.basename(img_path)

def process_image_grounding_paths(data, crop_root, proto_root):
    """Copy images to prototypes with crop subdir pattern and update paths. Handles list of lists. Returns relative path from prototypes dir."""
    def process(paths):
        # If paths is a list of lists, process each sublist
        if isinstance(paths, list) and paths and isinstance(paths[0], list):
            return [process(sublist) for sublist in paths]
        new_paths = []
        for img_path in paths:
            rel_path = get_relative_crop_path(img_path, crop_root)
            target_path = os.path.join(proto_root, rel_path)
            target_dir = os.path.dirname(target_path)
            if not os.path.exists(target_dir):
                os.makedirs(target_dir)
            try:
                shutil.copy(img_path, target_path)
            except Exception as e:
                print(f"Warning: Could not copy {img_path} to {target_path}: {e}")
            # Always return path as 'prototypes/...' (relative to workspace)
            new_paths.append(os.path.join("prototypes", rel_path))
        return new_paths

    if isinstance(data, dict):
        for k, v in data.items():
            if k == "image_grounding_paths":
                data[k] = process(v)
            else:
                process_image_grounding_paths(v, crop_root, proto_root)
    elif isinstance(data, list):
        for item in data:
            process_image_grounding_paths(item, crop_root, proto_root)
    return data

def convert_pth_to_json(pth_path, json_path, crop_root, proto_root):
    print(f"Loading {pth_path} ...")
    data = torch.load(pth_path, map_location="cpu")

    print("Converting tensors to JSON-serializable format ...")
    clean_data = tensor_to_list(data)

    print("Processing image_grounding_paths ...")
    clean_data = process_image_grounding_paths(clean_data, crop_root, proto_root)

    print(f"Saving to {json_path} ...")
    with open(json_path, "w") as f:
        json.dump(clean_data, f, indent=2)

    print("✅ Conversion complete!")

# Example usage
file_path = concept_pth
# Save JSON in prototypes_path directory
json_filename = os.path.splitext(os.path.basename(file_path))[0] + ".json"
output_path = os.path.join(front_end_output, json_filename)
convert_pth_to_json(file_path, output_path, image_crop_path, prototypes_path)





Loading /mnt/abka03/Projects/xl-vlms/outputs/1000_imagenet_cdgl/concept/snmf/combined_concept_snmf_raw.pth ...
Converting tensors to JSON-serializable format ...
Processing image_grounding_paths ...
Saving to /mnt/abka03/Projects/xl-vlms/outputs/1000_imagenet_cdgl/frontend/combined_concept_snmf_raw.json ...
Saving to /mnt/abka03/Projects/xl-vlms/outputs/1000_imagenet_cdgl/frontend/combined_concept_snmf_raw.json ...
✅ Conversion complete!
✅ Conversion complete!


In [24]:
# Improved check: use workspace root to get absolute path for each file listed under 'image_grounding_paths'
import os
import json

front_end_output_root = "/mnt/abka03/Projects/xl-vlms/outputs/1000_imagenet_cdgl/frontend"  # Set this to your workspace root

def check_image_grounding_paths_exist_absolute(json_path, front_end_output_root):
    with open(json_path, 'r') as f:
        data = json.load(f)

    missing_files = []

    def check_paths(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == 'image_grounding_paths':
                    paths_to_check = v if isinstance(v, list) else [v]
                    for sublist in paths_to_check:
                        if isinstance(sublist, list):
                            for rel_path in sublist:
                                abs_path = os.path.join(front_end_output_root, rel_path)
                                if not os.path.exists(abs_path):
                                    missing_files.append(rel_path)
                        else:
                            abs_path = os.path.join(front_end_output_root, sublist)
                            if not os.path.exists(abs_path):
                                missing_files.append(sublist)
                else:
                    check_paths(v)
        elif isinstance(obj, list):
            for item in obj:
                check_paths(item)

    check_paths(data)

    if missing_files:
        print(f"Missing files ({len(missing_files)}):")
        for path in missing_files:
            print(path)
    else:
        print("All files listed under 'image_grounding_paths' exist.")

# Example usage
json_path = os.path.join(front_end_output, os.path.splitext(os.path.basename(concept_pth))[0] + ".json")
check_image_grounding_paths_exist_absolute(json_path, front_end_output_root)


All files listed under 'image_grounding_paths' exist.


In [ ]:
prediction =f"{xai_output}/explanations/snmf/vlm_explanations.json"


In [25]:
# Update prediction JSON: replace image_grounding_path entries with prototype-relative paths, copy image_path to frontend/input, and update its path
import os
import json
import shutil

prediction_path = f"{xai_output}/explanations/snmf/vlm_explanations.json"
frontend_dir = front_end_output  # Save in frontend directory
frontend_input_dir = os.path.join(frontend_dir, "input")
prototypes_dir = os.path.join(frontend_dir, "prototypes")

os.makedirs(frontend_input_dir, exist_ok=True)

with open(prediction_path, "r") as f:
    pred_data = json.load(f)

for result in pred_data.get("results", []):
    # Update image_path: copy to frontend/input and set relative path
    orig_img_path = result.get("image_path")
    if orig_img_path:
        img_filename = os.path.basename(orig_img_path)
        input_img_path = os.path.join(frontend_input_dir, img_filename)
        try:
            shutil.copy(orig_img_path, input_img_path)
        except Exception as e:
            print(f"Warning: Could not copy {orig_img_path} to {input_img_path}: {e}")
        result["image_path"] = os.path.join("input", img_filename)

    # Update image_grounding_path entries in per_token_concepts
    for token in result.get("per_token_concepts", []):
        for concept in token.get("top_concepts", []):
            img_grounding_str = concept.get("image_grounding_path")
            if img_grounding_str:
                try:
                    img_list = eval(img_grounding_str)
                except Exception:
                    img_list = []
                new_img_list = []
                for abs_path in img_list:
                    parts = os.path.normpath(abs_path).split(os.sep)
                    if 'crops' in parts:
                        crops_idx = parts.index('crops')
                        rel_path = os.path.join(*parts[crops_idx+1:])
                        proto_path = os.path.join("prototypes", rel_path)
                        new_img_list.append(proto_path)
                    else:
                        new_img_list.append(os.path.basename(abs_path))
                concept["image_grounding_path"] = new_img_list

    # Update image_grounding_path entries in top_concepts_over_sequence
    for concept in result.get("top_concepts_over_sequence", []):
        img_grounding_str = concept.get("image_grounding_path")
        if img_grounding_str:
            try:
                img_list = eval(img_grounding_str)
            except Exception:
                img_list = []
            new_img_list = []
            for abs_path in img_list:
                parts = os.path.normpath(abs_path).split(os.sep)
                if 'crops' in parts:
                    crops_idx = parts.index('crops')
                    rel_path = os.path.join(*parts[crops_idx+1:])
                    proto_path = os.path.join("prototypes", rel_path)
                    new_img_list.append(proto_path)
                else:
                    new_img_list.append(os.path.basename(abs_path))
            concept["image_grounding_path"] = new_img_list

# Save updated prediction JSON in frontend directory
updated_pred_path = os.path.join(frontend_dir, "vlm_explanations_frontend.json")
with open(updated_pred_path, "w") as f:
    json.dump(pred_data, f, indent=2)

print(f"✅ Updated prediction JSON saved to {updated_pred_path}")


✅ Updated prediction JSON saved to /mnt/abka03/Projects/xl-vlms/outputs/1000_imagenet_cdgl/frontend/vlm_explanations_frontend.json
